# 01. Obtencion de la data

Esta notebook documenta la etapa de adquisicion de datos del caso de estudio. En esta version curada se conservan dos entradas principales:

- tablas resumidas de vacantes por ubicacion y tecnologia
- dataset survey de Stack Overflow usado como base del analisis principal

## Alcance

- establecer rutas reproducibles dentro del proyecto
- cargar y validar las fuentes de entrada
- consolidar las tablas de vacantes en un unico `CSV`
- dejar lista la fuente survey para limpieza, EDA, visualizaciones y dashboarding


## Subpaso 1. Configuracion de entorno

**Proposito:** establecer rutas reproducibles para trabajar desde la carpeta principal del proyecto.

**Resultado esperado:** acceso a `assets/source_tables/`, al archivo `data/survey_data_updated.csv` y a la salida final `data/job_postings_summary.csv`.


In [ ]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SOURCE_TABLES_DIR = PROJECT_ROOT / "assets" / "source_tables"
LOCATION_XLSX_PATH = SOURCE_TABLES_DIR / "job_postings_by_location.xlsx"
TECHNOLOGY_XLSX_PATH = SOURCE_TABLES_DIR / "job_postings_by_technology.xlsx"
JOB_POSTINGS_SUMMARY_PATH = DATA_DIR / "job_postings_summary.csv"
SURVEY_CSV_PATH = DATA_DIR / "survey_data_updated.csv"

def read_excel_basic(path: Path) -> pd.DataFrame:
    workbook = load_workbook(path, data_only=True)
    sheet = workbook.active
    rows = list(sheet.iter_rows(values_only=True))
    headers = rows[0]
    data = rows[1:]
    return pd.DataFrame(data, columns=headers)

LOCATION_XLSX_PATH, TECHNOLOGY_XLSX_PATH, SURVEY_CSV_PATH, JOB_POSTINGS_SUMMARY_PATH


## Fuente A. Vacantes laborales resumidas

La senal de demanda laboral se conserva en dos tablas pequenas: una por ubicacion y otra por tecnologia. En esta version del repositorio ambas se consolidan en un solo archivo final para reducir ruido en `data/` y dejar una entrada de mercado mas compacta.


## Subpaso 2. Carga de tablas base de vacantes

**Proposito:** cargar las dos tablas fuente que resumen la demanda observada.

**Resultado:** quedan disponibles una tabla por ubicacion y otra por tecnologia, listas para validacion y consolidacion.


In [ ]:
location_source = read_excel_basic(LOCATION_XLSX_PATH)
technology_source = read_excel_basic(TECHNOLOGY_XLSX_PATH)

print("Location summary shape:", location_source.shape)
print("Technology summary shape:", technology_source.shape)
location_source.head()


## Subpaso 3. Estandarizacion y union de las tablas de vacantes

**Proposito:** dejar explicita en codigo la consolidacion de `job_postings_by_location.xlsx` y `job_postings_by_technology.xlsx` en un unico archivo reutilizable.

**Resultado:** ambas tablas quedan homologadas a un mismo esquema y se unen en `data/job_postings_summary.csv`.

**Hallazgo operativo:** un formato largo con `summary_type`, `category` y `job_postings` simplifica el consumo posterior en scripts y visualizaciones.


In [ ]:
location_summary = (
    location_source.rename(
        columns={
            "Location": "category",
            "Number of Job Postings": "job_postings",
        }
    )
    .assign(summary_type="location")
    [["summary_type", "category", "job_postings"]]
)

technology_summary = (
    technology_source.rename(
        columns={
            "Technology": "category",
            "Number of Job Postings": "job_postings",
        }
    )
    .assign(summary_type="technology")
    [["summary_type", "category", "job_postings"]]
)

job_postings_summary = pd.concat(
    [location_summary, technology_summary],
    ignore_index=True,
)
job_postings_summary.to_csv(JOB_POSTINGS_SUMMARY_PATH, index=False)

print("Saved:", JOB_POSTINGS_SUMMARY_PATH)
job_postings_summary.head(10)


## Fuente B. Dataset survey de Stack Overflow

Esta es la fuente central del proyecto. En la carpeta `data/` se conserva `survey_data_updated.csv` como snapshot de entrada para limpieza, EDA, visualizaciones y dashboarding.


## Subpaso 4. Carga del dataset survey

**Proposito:** incorporar el dataset principal de la encuesta para el resto del flujo analitico.

**Resultado:** se carga una tabla amplia con respuestas de desarrolladores, variables demograficas, stack tecnologico, compensacion y experiencia laboral.


In [ ]:
survey_df = pd.read_csv(SURVEY_CSV_PATH)
survey_df[["ResponseId", "MainBranch", "Age", "Employment", "Country"]].head()


## Subpaso 5. Validacion rapida del dataset survey

**Proposito:** revisar tamano, estructura y primeras senales de complejidad antes de pasar a limpieza.

**Resultado:** el archivo conserva una estructura amplia y suficiente para sostener las siguientes etapas del proyecto.

**Hallazgos:** la combinacion de columnas demograficas, salariales y multiseleccion justifica una etapa de limpieza dedicada antes del EDA.


In [ ]:
survey_schema_summary = pd.DataFrame(
    {
        "dtype": survey_df.dtypes.astype(str),
        "missing_values": survey_df.isna().sum(),
        "non_null_count": survey_df.count(),
    }
)

print("Shape:", survey_df.shape)
print("First 20 columns:", survey_df.columns[:20].tolist())
survey_schema_summary.head(20)


## Cierre de la etapa

Esta etapa deja lista una senal compacta de mercado en `job_postings_summary.csv` y mantiene el survey raw como base del analisis principal. La siguiente notebook se concentra en limpieza de datos y preparacion del dataset final que se usara en EDA, visualizaciones y dashboarding.
